# 1 · Processamento de Dados — iFood Offer Optimization (PySpark)

**Objetivo do notebook:** transformar os três arquivos brutos (`offers`, `profile`,
`transactions`) em uma **tabela unificada de modelagem**, onde cada linha é uma
*oferta enviada a um cliente* com o tratamento e o desfecho necessários para
**uplift modeling**.

## Enquadramento do problema
A pergunta de negócio — *"qual oferta enviar para cada cliente?"* — é causal: só
vale a pena enviar uma oferta se ela **muda o comportamento** do cliente. Por isso
não modelamos apenas *quem completa* a oferta, e sim o **efeito incremental** de
expor o cliente à oferta.

Como o dataset não tem um grupo de controle "sem oferta", usamos um **experimento
quase-natural** já presente nos dados: nem todo cliente que **recebe** a oferta
efetivamente a **visualiza** (não abriu o canal). Definimos então:

- **Tratamento (`W`) = a oferta foi *visualizada* dentro da validade** (1) vs
  recebida mas não visualizada (0).
- **Desfecho (`Y`)**:
  - `bogo` / `discount` → a oferta foi **completada** dentro da validade;
  - `informational` (não tem "completar") → o cliente fez **alguma transação**
    dentro da validade.

O **uplift** de uma oferta para um cliente é `P(Y|W=1) − P(Y|W=0)`.

## Premissas (documentadas para o avaliador)
1. `age == 118` é um *placeholder* de idade ausente (coincide 100% com `gender` e
   `credit_card_limit` nulos) → tratado como perfil incompleto.
2. Janela de validade de uma oferta = `[t_recebida, t_recebida + duration]` (dias).
3. Visualização/conclusão são atribuídas a uma oferta recebida se ocorrem na
   janela e são do **mesmo `offer_id`**.
4. `time_since_test_start` está em **dias** (0–29.75; passos de 0.25 = 6 h).
5. Features comportamentais do cliente são agregadas sobre todo o período de teste;
   em produção seriam calculadas em uma janela estritamente anterior à oferta
   (registrado como limitação para evitar vazamento).
6. Se o mesmo `offer_id` é recebido mais de uma vez com janelas sobrepostas, um
   evento pode ser atribuído a mais de uma instância (double-count aceito).


## Setup

In [1]:
import os, sys
sys.path.append(os.path.abspath('..'))  # tornar 'src' importável a partir de notebooks/

from pyspark.sql import functions as F, Window
from src.config import (get_spark, OFFERS_JSON, PROFILE_JSON, TRANSACTIONS_JSON,
                        CUSTOMERS_PARQUET, OFFERS_PARQUET, MODELING_TABLE_PARQUET,
                        DATA_PROCESSED)

spark = get_spark("ifood-1-data-processing")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
print("Spark", spark.version)

C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark 4.2.0


## 1. Ofertas (`offers.json`)
Metadados das 10 ofertas. Explodimos a lista de canais em flags booleanas.

In [2]:
offers_raw = spark.read.json(str(OFFERS_JSON), multiLine=True)
offers_raw.show(truncate=False)

CHANNELS = ["web", "email", "mobile", "social"]
offers = offers_raw
for ch in CHANNELS:
    offers = offers.withColumn(f"ch_{ch}", F.array_contains("channels", ch).cast("int"))
offers = (offers
    .withColumn("n_channels", sum(F.col(f"ch_{ch}") for ch in CHANNELS))
    .withColumnRenamed("id", "offer_id")
    .withColumn("duration", F.col("duration").cast("double")))
offers.select("offer_id", "offer_type", "min_value", "discount_value",
              "duration", "n_channels", *[f"ch_{c}" for c in CHANNELS]).show(truncate=False)

+----------------------------+--------------+--------+--------------------------------+---------+-------------+
|channels                    |discount_value|duration|id                              |min_value|offer_type   |
+----------------------------+--------------+--------+--------------------------------+---------+-------------+
|[email, mobile, social]     |10            |7.0     |ae264e3637204a6fb9bb56bc8210ddfd|10       |bogo         |
|[web, email, mobile, social]|10            |5.0     |4d5c57ea9a6940dd891ad53e9dbe8da0|10       |bogo         |
|[web, email, mobile]        |0             |4.0     |3f207df678b143eea3cee63160fa8bed|0        |informational|
|[web, email, mobile]        |5             |7.0     |9b98b8c7a33c4b65b9aebfe6a799e6d9|5        |bogo         |
|[web, email]                |5             |10.0    |0b1e1539f2cc45b7b9fa7c272da2e1d7|20       |discount     |
|[web, email, mobile, social]|3             |7.0     |2298d6c36e964ae4a3e7e9706d1fb8c2|7        |discoun

+--------------------------------+-------------+---------+--------------+--------+----------+------+--------+---------+---------+
|offer_id                        |offer_type   |min_value|discount_value|duration|n_channels|ch_web|ch_email|ch_mobile|ch_social|
+--------------------------------+-------------+---------+--------------+--------+----------+------+--------+---------+---------+
|ae264e3637204a6fb9bb56bc8210ddfd|bogo         |10       |10            |7.0     |3         |0     |1       |1        |1        |
|4d5c57ea9a6940dd891ad53e9dbe8da0|bogo         |10       |10            |5.0     |4         |1     |1       |1        |1        |
|3f207df678b143eea3cee63160fa8bed|informational|0        |0             |4.0     |3         |1     |1       |1        |0        |
|9b98b8c7a33c4b65b9aebfe6a799e6d9|bogo         |5        |5             |7.0     |3         |1     |1       |1        |0        |
|0b1e1539f2cc45b7b9fa7c272da2e1d7|discount     |20       |5             |10.0    |2       

## 2. Perfil dos clientes (`profile.json`)
Limpeza de idade *placeholder*, parsing da data de cadastro e criação de features estáticas.

In [3]:
profile_raw = spark.read.json(str(PROFILE_JSON), multiLine=True)
print("linhas:", profile_raw.count())

# t=0 do teste como referência para calcular 'tenure' (a data absoluta é desconhecida;
# usamos a data de cadastro mais recente do dataset como proxy de 'hoje').
ref = profile_raw.select(F.max(F.to_date("registered_on", "yyyyMMdd")).alias("m")).first()["m"]

customers = (profile_raw
    .withColumnRenamed("id", "account_id")
    .withColumn("incomplete_profile", (F.col("age") == 118).cast("int"))
    .withColumn("age", F.when(F.col("age") == 118, None).otherwise(F.col("age")))
    .withColumn("registered_dt", F.to_date("registered_on", "yyyyMMdd"))
    .withColumn("tenure_days", F.datediff(F.lit(ref), F.col("registered_dt")))
    .withColumn("gender", F.when(F.col("gender").isNull(), "unknown").otherwise(F.col("gender"))))

customers.select("account_id","age","gender","credit_card_limit",
                 "tenure_days","incomplete_profile").show(5)
print("perfis incompletos:", customers.filter("incomplete_profile=1").count())

linhas: 17000


+--------------------+----+-------+-----------------+-----------+------------------+
|          account_id| age| gender|credit_card_limit|tenure_days|incomplete_profile|
+--------------------+----+-------+-----------------+-----------+------------------+
|68be06ca386d4c319...|NULL|unknown|             NULL|        529|                 1|
|0610b486422d4921a...|  55|      F|         112000.0|        376|                 0|
|38fe809add3b4fcf9...|NULL|unknown|             NULL|         14|                 1|
|78afa995795e4d85b...|  75|      F|         100000.0|        443|                 0|
|a03223e636434f42a...|NULL|unknown|             NULL|        356|                 1|
+--------------------+----+-------+-----------------+-----------+------------------+
only showing top 5 rows


perfis incompletos: 2175


## 3. Transações (`transactions.json`)
O campo `value` mistura chaves (`offer id` com espaço em received/viewed, `offer_id` com underscore em completed, `amount` em transações). Normalizamos tudo.

In [4]:
tx_raw = spark.read.json(str(TRANSACTIONS_JSON), multiLine=True)
tx_raw.printSchema()

tx = (tx_raw
    .withColumn("offer_id", F.coalesce(F.col("value.`offer id`"), F.col("value.offer_id")))
    .withColumn("amount", F.col("value.amount").cast("double"))
    .withColumn("reward", F.col("value.reward").cast("double"))
    .withColumnRenamed("time_since_test_start", "t")
    .select("account_id", "event", "offer_id", "amount", "reward", "t"))

tx.groupBy("event").count().orderBy(F.desc("count")).show()

root
 |-- account_id: string (nullable = true)
 |-- event: string (nullable = true)
 |-- time_since_test_start: double (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- amount: double (nullable = true)
 |    |-- offer id: string (nullable = true)
 |    |-- offer_id: string (nullable = true)
 |    |-- reward: double (nullable = true)



+---------------+------+
|          event| count|
+---------------+------+
|    transaction|138953|
| offer received| 76277|
|   offer viewed| 57725|
|offer completed| 33579|
+---------------+------+



### Separar os tipos de evento

In [5]:
received  = tx.filter("event='offer received'").select("account_id","offer_id",F.col("t").alias("t_recv"))
viewed    = tx.filter("event='offer viewed'").select("account_id","offer_id",F.col("t").alias("t_view"))
completed = tx.filter("event='offer completed'").select("account_id","offer_id",
                                                        F.col("t").alias("t_comp"),
                                                        F.col("reward").alias("reward_paid"))
transactions = tx.filter("event='transaction'").select("account_id",F.col("t").alias("t_tx"),"amount")

for name, d in [("received",received),("viewed",viewed),("completed",completed),("transactions",transactions)]:
    print(f"{name:13s}: {d.count():>7,}")

received     :  76,277


viewed       :  57,725


completed    :  33,579


transactions : 138,953


## 4. Construção da tabela de instâncias de oferta (coração do pipeline)
Cada **oferta recebida** vira uma instância com id único. Anexamos janela de
validade e, por *joins* com filtro de janela, marcamos visualização, conclusão e
gasto no período.

In [6]:
# instância única por oferta recebida
w_inst = Window.orderBy("account_id","offer_id","t_recv")
inst = (received
    .join(offers.select("offer_id","offer_type","duration","min_value",
                        "discount_value","n_channels",*[f"ch_{c}" for c in CHANNELS]),
          "offer_id", "left")
    .withColumn("t_end", F.col("t_recv") + F.col("duration"))
    .withColumn("instance_id", F.row_number().over(w_inst)))
print("instâncias de oferta:", inst.count())

instâncias de oferta: 76277


In [7]:
# --- Tratamento: oferta VISUALIZADA na janela ---
viewed_flag = (inst.select("instance_id","account_id","offer_id","t_recv","t_end")
    .join(viewed, ["account_id","offer_id"])
    .filter((F.col("t_view") >= F.col("t_recv")) & (F.col("t_view") <= F.col("t_end")))
    .groupBy("instance_id").agg(F.lit(1).alias("viewed")))

# --- Desfecho bogo/discount: oferta COMPLETADA na janela ---
completed_flag = (inst.select("instance_id","account_id","offer_id","t_recv","t_end")
    .join(completed, ["account_id","offer_id"])
    .filter((F.col("t_comp") >= F.col("t_recv")) & (F.col("t_comp") <= F.col("t_end")))
    .groupBy("instance_id").agg(F.lit(1).alias("completed"),
                                F.max("reward_paid").alias("reward_paid")))

# --- Gasto do cliente na janela (qualquer transação) ---
spend_win = (inst.select("instance_id","account_id","t_recv","t_end")
    .join(transactions, "account_id")
    .filter((F.col("t_tx") >= F.col("t_recv")) & (F.col("t_tx") <= F.col("t_end")))
    .groupBy("instance_id").agg(F.coalesce(F.sum("amount"),F.lit(0.0)).alias("spend_window"),
                                F.count("*").alias("n_tx_window")))
print("ok")

ok


In [8]:
instances = (inst
    .join(viewed_flag, "instance_id", "left")
    .join(completed_flag, "instance_id", "left")
    .join(spend_win, "instance_id", "left")
    .fillna({"viewed":0,"completed":0,"reward_paid":0.0,"spend_window":0.0,"n_tx_window":0}))

# Desfecho unificado Y: completado (bogo/discount) OU transacionou (informational)
instances = instances.withColumn(
    "Y",
    F.when(F.col("offer_type")=="informational", (F.col("n_tx_window")>0).cast("int"))
     .otherwise(F.col("completed")))
instances = instances.withColumnRenamed("viewed","W")  # W = tratamento (visualizou)

instances.select("offer_type","W","Y","completed","spend_window","reward_paid").show(5)

+-------------+---+---+---------+------------------+-----------+
|   offer_type|  W|  Y|completed|      spend_window|reward_paid|
+-------------+---+---+---------+------------------+-----------+
|         bogo|  1|  1|        1|             54.04|        5.0|
|         bogo|  1|  1|        1|             17.24|       10.0|
|informational|  0|  1|        0|51.099999999999994|        0.0|
|     discount|  1|  1|        1|             46.59|        3.0|
|     discount|  1|  1|        1|            156.18|        2.0|
+-------------+---+---+---------+------------------+-----------+
only showing top 5 rows


### Sanity-check do funil (deve bater com a EDA em pandas)

In [9]:
bd = instances.filter("offer_type in ('bogo','discount')")
tot = bd.count()
print(f"bogo/discount instâncias: {tot:,}")
print("taxa visualização :", round(bd.agg(F.mean('W')).first()[0],3))
print("taxa conclusão    :", round(bd.agg(F.mean('completed')).first()[0],3))
waste = bd.filter("completed=1 and W=0").count()/tot
print("COMPLETADAS SEM VER (recompensa desperdiçada):", round(waste,3))
up = (bd.filter('W=1').agg(F.mean('completed')).first()[0]
      - bd.filter('W=0').agg(F.mean('completed')).first()[0])
print("uplift naïve (viu - não viu):", round(up,3))

bogo/discount instâncias: 61,042


taxa visualização : 0.769


taxa conclusão    : 0.551


COMPLETADAS SEM VER (recompensa desperdiçada): 0.094


uplift naïve (viu - não viu): 0.188


## 5. Features comportamentais do cliente (RFM-like)
Agregadas sobre todas as transações do período. *Limitação:* incluem o período das
ofertas; em produção usaríamos janela estritamente anterior (premissa 5).

In [10]:
behav = (transactions.groupBy("account_id").agg(
            F.count("*").alias("n_transactions"),
            F.round(F.sum("amount"),2).alias("total_spend"),
            F.round(F.avg("amount"),2).alias("avg_ticket"),
            F.countDistinct(F.floor("t_tx")).alias("active_days"),
            F.max("t_tx").alias("last_tx_day")))
customers_full = (customers.join(behav, "account_id", "left")
    .fillna({"n_transactions":0,"total_spend":0.0,"avg_ticket":0.0,
             "active_days":0,"last_tx_day":0.0}))
customers_full.select("account_id","age","gender","credit_card_limit","tenure_days",
                      "n_transactions","total_spend","avg_ticket","active_days").show(5)

+--------------------+----+-------+-----------------+-----------+--------------+-----------+----------+-----------+
|          account_id| age| gender|credit_card_limit|tenure_days|n_transactions|total_spend|avg_ticket|active_days|
+--------------------+----+-------+-----------------+-----------+--------------+-----------+----------+-----------+
|38fe809add3b4fcf9...|NULL|unknown|             NULL|         14|             6|       14.3|      2.38|          6|
|e2127556f4f64592b...|  68|      M|          70000.0|         91|             3|      57.73|     19.24|          2|
|0610b486422d4921a...|  55|      F|         112000.0|        376|             3|      77.01|     25.67|          3|
|78afa995795e4d85b...|  75|      F|         100000.0|        443|             7|     159.27|     22.75|          7|
|a03223e636434f42a...|NULL|unknown|             NULL|        356|             3|       4.65|      1.55|          3|
+--------------------+----+-------+-----------------+-----------+-------

## 6. Tabela de modelagem unificada
Uma linha por instância de oferta = features do cliente + features da oferta + `W` + `Y` + economia.

In [11]:
feat_cols = ["age","gender","credit_card_limit","tenure_days","incomplete_profile",
             "n_transactions","total_spend","avg_ticket","active_days","last_tx_day"]
offer_cols = ["offer_type","min_value","discount_value","duration","n_channels",
              *[f"ch_{c}" for c in CHANNELS]]

modeling = (instances
    .join(customers_full.select("account_id",*feat_cols), "account_id", "left")
    .select("instance_id","account_id","offer_id",*offer_cols,*feat_cols,
            "W","Y","completed","spend_window","reward_paid","t_recv"))
print("tabela de modelagem:", modeling.count(), "linhas x", len(modeling.columns), "colunas")
modeling.show(4)

tabela de modelagem: 76277 linhas x 28 colunas


+-----------+--------------------+--------------------+-------------+---------+--------------+--------+----------+------+--------+---------+---------+---+------+-----------------+-----------+------------------+--------------+-----------+----------+-----------+-----------+---+---+---------+------------------+-----------+------+
|instance_id|          account_id|            offer_id|   offer_type|min_value|discount_value|duration|n_channels|ch_web|ch_email|ch_mobile|ch_social|age|gender|credit_card_limit|tenure_days|incomplete_profile|n_transactions|total_spend|avg_ticket|active_days|last_tx_day|  W|  Y|completed|      spend_window|reward_paid|t_recv|
+-----------+--------------------+--------------------+-------------+---------+--------------+--------+----------+------+--------+---------+---------+---+------+-----------------+-----------+------------------+--------------+-----------+----------+-----------+-----------+---+---+---------+------------------+-----------+------+
|         12|

## 7. Persistência (Parquet)
Materializamos via `pandas`/`pyarrow` (o dataset cabe em memória). Isso mantém
todas as *transformações* em Spark e evita a dependência de `winutils.exe` que o
gravador Hadoop do Spark exige no Windows.

In [12]:
customers_full.toPandas().to_parquet(CUSTOMERS_PARQUET, index=False)
offers.toPandas().to_parquet(OFFERS_PARQUET, index=False)
modeling.toPandas().to_parquet(MODELING_TABLE_PARQUET, index=False)
print("Salvo em", DATA_PROCESSED)
for p in [CUSTOMERS_PARQUET, OFFERS_PARQUET, MODELING_TABLE_PARQUET]:
    print(" -", p.name)

C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\Maxel\miniconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Salvo em C:\Users\Maxel\OneDrive\Documentos\github\ifood-case\data\processed
 - customers.parquet
 - offers.parquet
 - modeling_table.parquet


## Resumo
- **Entrada:** 10 ofertas, 17.000 clientes, ~306k eventos.
- **Saída:** `modeling_table` (uma linha por oferta enviada) pronta para uplift,
  com tratamento (`W` = visualizou), desfecho (`Y`) e variáveis de economia
  (`spend_window`, `reward_paid`).
- **Achados-chave para o negócio:** ~9% das ofertas são **completadas sem serem
  vistas** (recompensa desperdiçada) e o uplift de *discount* supera muito o de
  *bogo* — base para a estratégia de segmentação do notebook 2.

In [13]:
spark.stop()